# 01 - Data Cleaning Report

**Project:** LaptopLens, Gaming Laptop Market Intelligence Pipeline
**Phase:** 1 (Data Quality & Reliability)
**Input:** `data/enriched_products.csv` (scraped from idefix.com, specs extracted with Gemini)
**Output:** `data/cleaned_products.csv`

## Goal

Turn the enriched dataset into an analysis-ready one, and document every cleaning decision so the process is reproducible and defensible.

## Principles

- The raw data files are never modified; the cleaned version is saved separately.
- No imputation with statistics (mean/median) here. That belongs inside the modeling `Pipeline` in Phase 3 to avoid data leakage.

## Notes

- Data was collected on: *18.11.2026*
- Prices are in Turkish lira (TL) and reflect the day of collection.

## First look into Data
Here we don't analyze the data, but we only examine it and find the problems in the set.

Most common problems:
- Duplicates
- Missing Values (NaN)
- Wrong Labels (e.g. 0.0 for price)

In [182]:
import pandas as pd
import numpy as np

pd.set_option('display.max_columns', None)
df = pd.read_csv("../data/enriched_products.csv")
print("Number of rows: ", df.shape[0])
df.columns.tolist()

Number of rows:  1950


['Link',
 'Name',
 'Price',
 'brand',
 'model',
 'cpu_model',
 'ram_gb',
 'gpu_model',
 'vram_gb',
 'storage_gb',
 'refresh_rate_hz',
 'screen_size',
 'os']

In [183]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1950 entries, 0 to 1949
Data columns (total 13 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Link             1950 non-null   str    
 1   Name             1950 non-null   str    
 2   Price            1950 non-null   float64
 3   brand            1949 non-null   str    
 4   model            1949 non-null   str    
 5   cpu_model        1942 non-null   str    
 6   ram_gb           1943 non-null   float64
 7   gpu_model        1909 non-null   str    
 8   vram_gb          1389 non-null   float64
 9   storage_gb       1940 non-null   float64
 10  refresh_rate_hz  1721 non-null   float64
 11  screen_size      1909 non-null   float64
 12  os               1891 non-null   str    
dtypes: float64(6), str(7)
memory usage: 198.2 KB


In [184]:
nulls = df.isna().sum().to_frame("null_count")
nulls["percentage"] = nulls["null_count"] / df.shape[0] * 100
nulls

,null_count,percentage
Link,0,0.000000
Name,0,0.000000
Price,0,0.000000
brand,1,0.051282
model,1,0.051282
cpu_model,8,0.410256
ram_gb,7,0.358974
gpu_model,41,2.102564
vram_gb,561,28.769231
storage_gb,10,0.512821


In [185]:
num_cols = ["Price", "ram_gb", "vram_gb", "storage_gb", "refresh_rate_hz", "screen_size"]
(df[num_cols] == 0).sum()

Price              25
ram_gb              0
vram_gb             0
storage_gb          0
refresh_rate_hz     0
screen_size         0
dtype: int64

In [186]:
df[num_cols].describe()

,Price,ram_gb,vram_gb,storage_gb,refresh_rate_hz,screen_size
count,1950.000000,1943.000000,1389.000000,1940.000000,1721.000000,1909.000000
mean,155459.706523,36.226454,8.840893,1666.509278,177.754794,15.953903
std,92863.960975,27.098899,3.021756,1834.161036,39.923681,0.650054
min,0.000000,8.000000,4.000000,256.000000,120.000000,10.000000
25%,94559.450000,16.000000,8.000000,512.000000,144.000000,15.600000
50%,127966.135000,32.000000,8.000000,1024.000000,165.000000,16.000000
75%,197223.245000,48.000000,8.000000,2048.000000,180.000000,16.000000
max,652299.140000,128.000000,24.000000,8192.000000,300.000000,18.000000


In [187]:
print("Total number of rows: ", df.shape[0])
print("Unique number of links: ", df["Link"].nunique())
print("Unique number of names: ", df["Name"].nunique())

Total number of rows:  1950
Unique number of links:  1950
Unique number of names:  1934


## Data Cleaning

Here, we are cleaning the set by
- removing the duplicates
- dropping unknowns price value rows
- removing non-laptop products
- normalize GPU names
- fill VRAM with mod for that model

In [188]:
df_clean = df.copy(deep=True)
log = [("Initial rows", len(df_clean))]

In [189]:
df_clean["Price"] = df_clean["Price"].replace(0, np.nan)
df_clean = df_clean.dropna(subset=["Price"])
log.append(("Dropped rows with zero/missing price", len(df_clean)))
log[-1]

('Dropped rows with zero/missing price', 1925)

In [190]:
df_clean.nsmallest(10,"Price")[["Name", "Price"]]

,Name,Price
1872,Astraltech K98 OYUNCU KULAKLIĞI,1261.25
1117,ASUS TUF GAMING A2 USB3.2 IP68 HARICI M.2 SSD KUTUSU,4781.00
124,"ASUS Vivobook 15 X1504VA-NJ739 i3-1315U 8 GB 512 GB SSD UHD Graphics 15.6"" Full HD Notebook",29999.00
52,ASUS Vivobook 15 X1504VA-BQ3970W Intel Core I5 120U 8 GB 512GB SSD 15.6'' FHD Windows 11 Home,31999.00
1053,"ASUS Vivobook 15 A1504Ma-Bq695W Intel Core 3 304 8Gb 256Gb Ssd W11 Home 15.6"" Notebook Taşınabilir Bilgisayar",34761.82
107,Lenovo V15-G4 82YU0124TX AMD Ryzen 5-7520U 8GB 512GB AMD Radeon Graphics 15.6 inç Full HD FreeDos Laptop,36016.20
73,"Lenovo V15 G4 82YY0013TR Ryzen 7 7730U 16GB 512GB SSD 15.6"" Full HD FreeDOS Notebook",37462.00
1159,"HP OmniBook 3 16-BY0011NT1 AMD Ryzen 5 7520U 8GB 512GB SSD Freedos 16"" WUXGA IPS Taşınabilir Bilgisayar DY0G7EAA1",37823.22
1397,"HP OmniBook 3 16-BY0011NT2 AMD Ryzen 5 7520U 8GB 512GB SSD Windows 11 Home 16"" WUXGA IPS Taşınabilir Bilgisayar DY0G7EAA2",40077.98
1398,"HP OmniBook 3 16-BY0011NT4 AMD Ryzen 5 7520U 8GB 512GB SSD Windows 10 Home 16"" WUXGA IPS Taşınabilir Bilgisayar DY0G7EAA4",40077.98


In [191]:
df_clean[df_clean["gpu_model"].isna() & df_clean["screen_size"].isna()]

,Link,Name,Price,brand,model,cpu_model,ram_gb,gpu_model,vram_gb,storage_gb,refresh_rate_hz,screen_size,os
1114,https://www.idefix.com/deha-solutions-hp-omen-transcend-315-type-c-oled-9r181e9-p-33517403#sortId=33517403,Deha Solutions HP OMEN Transcend 31.5'' Type-C OLED (9R181E9),76999.00,HP,OMEN Transcend 31.5'' Type-C OLED (9R181E9),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1117,https://www.idefix.com/asus-tuf-gaming-a2-usb32-ip68-harici-m2-ssd-kutusu-p-33188845#sortId=33188845,ASUS TUF GAMING A2 USB3.2 IP68 HARICI M.2 SSD KUTUSU,4781.00,ASUS,TUF GAMING A2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1159,https://www.idefix.com/hp-omnibook-3-16-by0011nt1-amd-ryzen-5-7520u-8gb-512gb-ssd-freedos-16-wuxga-ips-tasinabilir-bilgisayar-dy0g7eaa1-p-33256484#sortId=33256484,"HP OmniBook 3 16-BY0011NT1 AMD Ryzen 5 7520U 8GB 512GB SSD Freedos 16"" WUXGA IPS Taşınabilir Bilgisayar DY0G7EAA1",37823.22,HP,OmniBook 3 16-BY0011NT1,AMD Ryzen 5 7520U,8.0,NaN,NaN,512.0,NaN,NaN,Free
1335,https://www.idefix.com/msi-nb-cyborg-15-b13wgkg-847xtr-p-33517422#sortId=33517422,MSI NB CYBORG 15 B13WGKG-847XTR,157710.00,MSI,NB CYBORG 15 B13WGKG-847XTR,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1872,https://www.idefix.com/astraltech-k98-oyuncu-kulakligi-p-31054446#sortId=31054446,Astraltech K98 OYUNCU KULAKLIĞI,1261.25,Astraltech,K98,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [192]:
accessory_idx = [1114, 1117, 1872]
df_clean = df_clean.drop(index=accessory_idx)
log.append(("Dropped non-laptop products (headset, SSD enclosure, monitor)", len(df_clean)))
log[-1]

('Dropped non-laptop products (headset, SSD enclosure, monitor)', 1922)

In [193]:
before = len(df_clean)
df_clean = df_clean.drop_duplicates(subset=["Name", "Price"])
print("Exact duplicates dropped:", before - len(df_clean))
log.append(("Dropped exact duplicates (same name and price)", len(df_clean)))

# Same product from different sellers stays, but shares a group_id (for grouped train/test split later)
df_clean["group_id"] = df_clean["Name"].str.upper().str.strip().factorize()[0]

Exact duplicates dropped: 0


In [194]:
import re
TR_MAP = str.maketrans({"İ": "I", "ı": "i"})  # Turkish dotted/dotless I breaks .upper() and regex

def normalize_gpu(raw):
    if pd.isna(raw):
        return np.nan
    s = str(raw).translate(TR_MAP).upper()

    m = re.search(r"\bRTX\s*PRO\s*(\d{3,4})\b", s)
    if m:
        return f"RTX Pro {m.group(1)}"

    m = re.search(r"\b(RTX|GTX)\s*-?\s*(\d{3,4})\s*(TI|SUPER)?\b", s)
    if m:
        series, number, suffix = m.groups()
        return f"{series} {number}" + (f" {suffix.title()}" if suffix else "")

    if re.search(r"RADEON|UHD|IRIS|\bARC\b|INTEL|AMD", s):
        return "Integrated"

    return str(raw).strip()

df_clean["gpu_model"] = df_clean["gpu_model"].apply(normalize_gpu)
log.append(("Normalized GPU names (gpu_clean)", len(df_clean)))
print(df_clean["gpu_model"].value_counts(dropna=False).to_string())

gpu_model
RTX 5060        590
RTX 5070        393
RTX 5050        301
RTX 4050        223
RTX 3050        148
RTX 5070 Ti     106
RTX 5080         88
NaN              33
RTX 5090         25
RTX 4060          6
Integrated        5
RTX Pro 3000      4


In [195]:
vram_mode = df_clean.groupby("gpu_model")["vram_gb"].agg(
    lambda s: s.mode().iloc[0] if s.notna().any() else np.nan
)
fill_values = df_clean["gpu_model"].map(vram_mode)

df_clean["vram_gb"] = df_clean["vram_gb"].fillna(fill_values)

print("VRAM mode per GPU:\n", vram_mode.to_string())
log.append(("Filled missing VRAM with per-GPU mode (flag: vram_imputed)", len(df_clean)))

VRAM mode per GPU:
 gpu_model
Integrated       NaN
RTX 3050         6.0
RTX 4050         6.0
RTX 4060         8.0
RTX 5050         8.0
RTX 5060         8.0
RTX 5070         8.0
RTX 5070 Ti     12.0
RTX 5080        16.0
RTX 5090        24.0
RTX Pro 3000    12.0


In [196]:
# 500 -> 512 (seller rounding); 1228 and 1638 are LLM TB->GB conversion errors (12TB, 16TB)
df_clean["storage_gb"] = df_clean["storage_gb"].replace({500: 512, 1228: 12288, 1638: 16384})
print(df_clean["storage_gb"].value_counts().sort_index().to_string())
log.append(("Fixed storage values (500->512, 1228->12288, 1638->16384)", len(df_clean)))

storage_gb
256.0      152
512.0      591
1024.0     538
2048.0     315
4096.0     230
8192.0      89
12288.0      1
16384.0      2


In [197]:
for col in ["brand", "model", "cpu_model"]:
    df_clean[col] = (
        df_clean[col].astype("string")
        .str.translate(TR_MAP)
        .str.upper()
        .str.strip()
        .str.replace(r"\s+", " ", regex=True)
    )

print(df_clean["brand"].value_counts().to_string())
print(df_clean["os"].value_counts(dropna=False).to_string())

brand
MSI            471
ASUS           387
LENOVO         361
ACER           243
CASPER         227
HP             161
GAMERAIDER      39
DELL            17
QUANTUM          7
THUNDEROBOT      5
GIGABYTE         1
MONSTER          1
TECHNOPC         1
QUADRO           1
os
Windows    1064
Free        807
NaN          50
Linux         1


In [198]:
before = len(df_clean)
df_clean = df_clean.dropna()
after = len(df_clean)
log.append((f"Removed NaN value rows ({before-after})", len(df_clean)))
log[-1]

('Removed NaN value rows (241)', 1681)

In [199]:
df_clean = df_clean.reset_index(drop=True)

In [200]:
df_clean.info()
df_clean.describe()

<class 'pandas.DataFrame'>
RangeIndex: 1681 entries, 0 to 1680
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Link             1681 non-null   str    
 1   Name             1681 non-null   str    
 2   Price            1681 non-null   float64
 3   brand            1681 non-null   string 
 4   model            1681 non-null   string 
 5   cpu_model        1681 non-null   string 
 6   ram_gb           1681 non-null   float64
 7   gpu_model        1681 non-null   str    
 8   vram_gb          1681 non-null   float64
 9   storage_gb       1681 non-null   float64
 10  refresh_rate_hz  1681 non-null   float64
 11  screen_size      1681 non-null   float64
 12  os               1681 non-null   str    
 13  group_id         1681 non-null   int64  
dtypes: float64(6), int64(1), str(4), string(3)
memory usage: 184.0 KB


,Price,ram_gb,vram_gb,storage_gb,refresh_rate_hz,screen_size,group_id
count,1681.000000,1681.000000,1681.000000,1681.000000,1681.000000,1681.000000,1681.000000
mean,165254.819851,38.298632,8.662701,1800.375967,178.035098,15.985723,960.480666
std,92350.281196,28.080307,2.864578,2011.980396,40.008652,0.630833,542.339512
min,43672.450000,8.000000,4.000000,256.000000,120.000000,14.000000,0.000000
25%,99999.000000,16.000000,8.000000,512.000000,144.000000,15.600000,501.000000
50%,135762.420000,32.000000,8.000000,1024.000000,165.000000,16.000000,921.000000
75%,207828.990000,48.000000,8.000000,2048.000000,180.000000,16.000000,1443.000000
max,652299.140000,128.000000,24.000000,16384.000000,300.000000,18.000000,1907.000000


In [201]:
df_clean.head()

,Link,Name,Price,brand,model,cpu_model,ram_gb,gpu_model,vram_gb,storage_gb,refresh_rate_hz,screen_size,os,group_id
0,https://www.idefix.com/asus-tuf-gaming-a15-fa506ncg-hn267-amd-ryzen-7-7445hs-16gb-512gb-ssd-rtx3050-4gb-75w-freedos-156-fhd-144hz-tasinabilir-p-16505594#sortId=16505594,"ASUS TUF Gaming A15 FA506NCG-HN267 AMD Ryzen 7 7445HS 16GB 512GB SSD RTX3050-4GB 75W Freedos 15.6"" FHD 144Hz Taşınabilir",43672.45,ASUS,TUF GAMING A15 FA506NCG-HN267,AMD RYZEN 7 7445HS,16.0,RTX 3050,4.0,512.0,144.0,15.6,Free,0
1,https://www.idefix.com/msi-vector-16-hx-ai-a2xwhg-091xtr-ultra-7-255hx-16gb-1tb-ssd-12gb-rtx5070ti-140w-16-fhd-144hz-free-p-8332470#sortId=8332470,MSI Vector 16 HX AI A2XWHG-091XTR Ultra 7 255HX 16GB 1TB SSD 12GB RTX5070Ti 140W 16 FHD+ 144Hz Free,124770.56,MSI,VECTOR 16 HX AI A2XWHG-091XTR,ULTRA 7 255HX,16.0,RTX 5070 Ti,12.0,1024.0,144.0,16.0,Free,1
2,https://www.idefix.com/asus-tuf-gaming-a16-fa607nug-rl213-ryzen-7-7445hs-16gb-ddr5-512gb-140w-rtx4050-6gb-16-144hz-dos-p-18542771#sortId=18542771,"ASUS TUF GAMING A16 FA607NUG-RL213 RYZEN 7-7445HS 16GB DDR5 512GB 140W RTX4050 6GB 16"" 144Hz DOS",69999.00,ASUS,TUF GAMING A16 FA607NUG-RL213,RYZEN 7-7445HS,16.0,RTX 4050,6.0,512.0,144.0,16.0,Free,3
3,https://www.idefix.com/hp-victus-gaming-laptop-intel-core-5-210h-16gb-1tb-ssd-rtx5060-144hz-dos-156-fhd-mika-gumus-c21sgea-p-17470137#sortId=17470137,"HP Victus Gaming Laptop Intel Core 5-210H 16GB 1TB SSD RTX5060 144Hz DOS 15.6"" FHD Mika Gümüş C21SGEA",68999.00,HP,VICTUS GAMING LAPTOP C21SGEA,INTEL CORE 5-210H,16.0,RTX 5060,8.0,1024.0,144.0,15.6,Free,5
4,https://www.idefix.com/msi-katana-17-hx-b14wfk-248xtr-i7-14650hx-16gb-1tb-ssd-8gb-rtx5060-115w-173-qhd-240hz-freedos-p-14942421#sortId=14942421,MSI Katana 17 HX B14WFK-248XTR i7-14650HX 16GB 1TB SSD 8GB RTX5060 115W 17.3 QHD 240Hz FreeDOS,96799.00,MSI,KATANA 17 HX B14WFK-248XTR,I7-14650HX,16.0,RTX 5060,8.0,1024.0,240.0,17.3,Free,6


## Save New Dataset

Now that the preprocessing phase is completed, we can save our clean dataset.

In [202]:
df_clean.to_csv("../data/cleaned_products.csv", index=False, encoding="utf-8-sig")
print(df_clean.isna().sum().to_string())
pd.DataFrame(log, columns=["step", "rows"])

Link               0
Name               0
Price              0
brand              0
model              0
cpu_model          0
ram_gb             0
gpu_model          0
vram_gb            0
storage_gb         0
refresh_rate_hz    0
screen_size        0
os                 0
group_id           0


,step,rows
0,Initial rows,1950
1,Dropped rows with zero/missing price,1925
2,"Dropped non-laptop products (headset, SSD enclosure, monitor)",1922
3,Dropped exact duplicates (same name and price),1922
4,Normalized GPU names (gpu_clean),1922
5,Filled missing VRAM with per-GPU mode (flag: vram_imputed),1922
6,"Fixed storage values (500->512, 1228->12288, 1638->16384)",1922
7,Removed NaN value rows (241),1681
